In [5]:
import os
import pandas as pd

base_path = "/kaggle/input/datasets/rishab260/uta-reallife-drowsiness-dataset"

data = []

valid_extensions = (".mov", ".mp4", ".avi", ".mkv")

for fold in os.listdir(base_path):
    fold_path = os.path.join(base_path, fold)

    if not os.path.isdir(fold_path):
        continue

    inner_path = os.path.join(fold_path, fold)

    if not os.path.isdir(inner_path):
        continue

    for subject in os.listdir(inner_path):
        subject_path = os.path.join(inner_path, subject)

        if not os.path.isdir(subject_path):
            continue

        for video in os.listdir(subject_path):

            video_clean = video.lower().strip()

            if not video_clean.endswith(valid_extensions):
                continue

            name = video_clean.split('.')[0]

            if name == "0":
                label = "Alert"
            elif name == "5":
                label = "Low Vigilance"
            elif name == "10":
                label = "Drowsy"
            else:
                print("Skipped (unknown):", video)
                continue

            data.append({
                "fold": fold,
                "subject": subject,
                "video": video,
                "label": label,
                "path": os.path.join(subject_path, video)
            })

df = pd.DataFrame(data)

print("✅ Total rows:", len(df))
display(df.head())

# ✅ 🔥 SAVE FILE (IMPORTANT)
save_path = "/kaggle/working/metadata_clean.csv"
df.to_csv(save_path, index=False)

print(f"✅ Metadata saved at: {save_path}")

Skipped (unknown): 10_1.mp4
Skipped (unknown): 10_2.mp4
✅ Total rows: 141


,fold,subject,video,label,path
0,Fold4_part2,47,5.mp4,Low Vigilance,/kaggle/input/datasets/rishab260/uta-reallife-...
1,Fold4_part2,47,10.mp4,Drowsy,/kaggle/input/datasets/rishab260/uta-reallife-...
2,Fold4_part2,47,0.mp4,Alert,/kaggle/input/datasets/rishab260/uta-reallife-...
3,Fold4_part2,45,5.mp4,Low Vigilance,/kaggle/input/datasets/rishab260/uta-reallife-...
4,Fold4_part2,45,10.mp4,Drowsy,/kaggle/input/datasets/rishab260/uta-reallife-...


✅ Metadata saved at: /kaggle/working/metadata_clean.csv


In [6]:
%pip install -q mediapipe opencv-python-headless scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 74.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [7]:
# If mediapipe is not installed, run this once:
# %pip install -q mediapipe opencv-python-headless scikit-learn tqdm

import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from tqdm import tqdm

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("mediapipe:", mp.__version__)
print("opencv:", cv2.__version__)

2026-05-01 05:12:30.858177: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777612351.135440      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777612351.208306      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777612351.878409      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777612351.878462      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777612351.878465      57 computation_placer.cc:177] computation placer alr

mediapipe: 0.10.35
opencv: 4.13.0


In [8]:
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task

print("Model downloaded ✅")

Model downloaded ✅


In [9]:
base_options = python.BaseOptions(
    model_asset_path="face_landmarker.task"
)

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False
)

detector = vision.FaceLandmarker.create_from_options(options)

print("FaceLandmarker ready ✅")

FaceLandmarker ready ✅


W0000 00:00:1777612376.952080     138 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1777612376.997356     140 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777612377.027249     140 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [10]:
# Eye landmarks for EAR
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

# Mouth landmarks for MAR
MOUTH_TOP = 13
MOUTH_BOTTOM = 14
MOUTH_LEFT = 78
MOUTH_RIGHT = 308

# Approximate head pose landmarks
NOSE = 1
CHIN = 152
LEFT_FACE = 234
RIGHT_FACE = 454

In [11]:
def compute_EAR(landmarks, eye):
    """
    EAR = Eye Aspect Ratio.
    Smaller EAR usually means eyes are closing.
    """
    p1, p2, p3, p4, p5, p6 = [np.array(landmarks[i][:2]) for i in eye]

    vertical1 = np.linalg.norm(p2 - p6)
    vertical2 = np.linalg.norm(p3 - p5)
    horizontal = np.linalg.norm(p1 - p4)

    return (vertical1 + vertical2) / (2.0 * horizontal + 1e-6)


def compute_MAR(landmarks):
    """
    MAR = Mouth Aspect Ratio.
    Larger MAR can indicate yawning/open mouth.
    """
    top = np.array(landmarks[MOUTH_TOP][:2])
    bottom = np.array(landmarks[MOUTH_BOTTOM][:2])
    left = np.array(landmarks[MOUTH_LEFT][:2])
    right = np.array(landmarks[MOUTH_RIGHT][:2])

    vertical = np.linalg.norm(top - bottom)
    horizontal = np.linalg.norm(left - right)

    return vertical / (horizontal + 1e-6)


def compute_head_pose_approx(landmarks):
    """
    Approximate head pose features.
    This is NOT true 3D pose, but useful as simple geometric features.
    """
    nose = np.array(landmarks[NOSE][:2])
    chin = np.array(landmarks[CHIN][:2])
    left = np.array(landmarks[LEFT_FACE][:2])
    right = np.array(landmarks[RIGHT_FACE][:2])

    pitch = np.linalg.norm(nose - chin)
    yaw = np.linalg.norm(left - right)

    return pitch, yaw

In [12]:
def get_face_landmarks_from_frame(frame_bgr):
    """
    Uses new MediaPipe Tasks API.
    Input: OpenCV BGR frame
    Output: list of pixel landmarks [(x, y, z), ...] or None
    """

    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = detector.detect(mp_image)

    if not result.face_landmarks:
        return None

    h, w, _ = frame_bgr.shape

    landmarks_px = []
    for lm in result.face_landmarks[0]:
        x = int(lm.x * w)
        y = int(lm.y * h)
        z = lm.z
        landmarks_px.append((x, y, z))

    return landmarks_px

In [13]:
def process_video(path, label, subject, sample_rate=10):
    """
    Extracts EAR, MAR, pitch, yaw from video frames using MediaPipe Tasks API.
    """

    cap = cv2.VideoCapture(path)

    frame_id = 0
    rows = []
    failed_frames = 0
    processed_frames = 0

    if not cap.isOpened():
        print("Could not open video:", path)
        return rows

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        frame_id += 1

        # Process every Nth frame
        if frame_id % sample_rate != 0:
            continue

        processed_frames += 1

        landmarks = get_face_landmarks_from_frame(frame)

        if landmarks is None:
            failed_frames += 1
            continue

        try:
            left_ear = compute_EAR(landmarks, LEFT_EYE)
            right_ear = compute_EAR(landmarks, RIGHT_EYE)
            EAR = (left_ear + right_ear) / 2.0

            MAR = compute_MAR(landmarks)
            pitch, yaw = compute_head_pose_approx(landmarks)

            rows.append({
                "subject": subject,
                "label": label,
                "frame": frame_id,
                "EAR": EAR,
                "MAR": MAR,
                "pitch": pitch,
                "yaw": yaw,
                "video_path": path
            })

        except Exception as e:
            failed_frames += 1
            # optional debug:
            # print("Feature extraction error:", e)

    cap.release()

    print(
        f"Done: {os.path.basename(path)} | "
        f"processed={processed_frames}, saved={len(rows)}, failed={failed_frames}"
    )

    return rows

In [14]:
df = pd.read_csv("/kaggle/working/metadata_clean.csv")

sample = df.iloc[0]

rows = process_video(
    sample["path"],
    sample["label"],
    sample["subject"],
    sample_rate=10
)

test_df = pd.DataFrame(rows)
print("Frames extracted:", len(test_df))
display(test_df.head())

Done: 5.mp4 | processed=1802, saved=1802, failed=0
Frames extracted: 1802


,subject,label,frame,EAR,MAR,pitch,yaw,video_path
0,47,Low Vigilance,10,0.295778,0.000000,258.414396,425.856783,/kaggle/input/datasets/rishab260/uta-reallife-...
1,47,Low Vigilance,20,0.287280,0.007511,242.596373,414.435761,/kaggle/input/datasets/rishab260/uta-reallife-...
2,47,Low Vigilance,30,0.285697,0.000000,243.392687,424.445521,/kaggle/input/datasets/rishab260/uta-reallife-...
3,47,Low Vigilance,40,0.296530,0.007654,238.581223,418.001196,/kaggle/input/datasets/rishab260/uta-reallife-...
4,47,Low Vigilance,50,0.283510,0.008033,240.520269,414.837318,/kaggle/input/datasets/rishab260/uta-reallife-...


In [15]:
import os
from tqdm import tqdm

save_dir = "/kaggle/working/features"
os.makedirs(save_dir, exist_ok=True)

for i, row in tqdm(df.iterrows(), total=len(df)):

    save_file = os.path.join(save_dir, f"features_{i}.csv")

    # ✅ skip already processed (important for resume)
    if os.path.exists(save_file):
        print(f"Skipping {i}, already done")
        continue

    try:
        rows = process_video(
            row["path"],
            row["label"],
            row["subject"],
            sample_rate=10
        )

        pd.DataFrame(rows).to_csv(save_file, index=False)

        print(f"✅ Saved: {save_file}")

    except Exception as e:
        print("❌ Error:", row["path"], e)

  1%|          | 1/141 [03:18<7:43:03, 198.45s/it]

Done: 5.mp4 | processed=1802, saved=1802, failed=0
✅ Saved: /kaggle/working/features/features_0.csv


  1%|▏         | 2/141 [06:46<7:52:26, 203.93s/it]

Done: 10.mp4 | processed=1834, saved=1834, failed=0
✅ Saved: /kaggle/working/features/features_1.csv


  2%|▏         | 3/141 [10:05<7:44:21, 201.90s/it]

...

Done: 5.mp4 | processed=1974, saved=1974, failed=0
✅ Saved: /kaggle/working/features/features_138.csv


 99%|█████████▉| 140/141 [5:18:42<02:42, 162.94s/it]

Done: 10.mp4 | processed=1869, saved=1869, failed=0
✅ Saved: /kaggle/working/features/features_139.csv


100%|██████████| 141/141 [5:23:54<00:00, 137.83s/it]

Done: 0.mp4 | processed=1838, saved=1838, failed=0
✅ Saved: /kaggle/working/features/features_140.csv


In [16]:
import glob

files = glob.glob("/kaggle/working/features/*.csv")

df_list = [pd.read_csv(f) for f in files]

features_df = pd.concat(df_list, ignore_index=True)

print("Total rows:", len(features_df))

Total rows: 235706


In [18]:
import glob
import pandas as pd
import os

# Folder where individual feature CSVs are saved
features_dir = "/kaggle/working/features"

# Output concatenated CSV file
output_file = "/kaggle/working/features_progress.csv"

# Get all CSV files
files = glob.glob(os.path.join(features_dir, "*.csv"))

print("Number of files found:", len(files))

# Read and concatenate
df_list = [pd.read_csv(f) for f in files]

features_df = pd.concat(df_list, ignore_index=True)

print("Total rows:", len(features_df))

# Save concatenated file
features_df.to_csv(output_file, index=False)

print(f"✅ Concatenated feature file saved at: {output_file}")

Number of files found: 141
Total rows: 235706
✅ Concatenated feature file saved at: /kaggle/working/features_progress.csv


In [19]:
features_df = pd.read_csv("/kaggle/working/features_progress.csv")

numeric_cols = ["EAR", "MAR", "pitch", "yaw"]

baseline = (
    features_df[features_df["label"] == "Alert"]
    .groupby("subject")[numeric_cols]
    .mean()
)

def normalize(row):
    subject = row["subject"]

    if subject in baseline.index:
        row["EAR_dev"] = row["EAR"] - baseline.loc[subject, "EAR"]
        row["MAR_dev"] = row["MAR"] - baseline.loc[subject, "MAR"]
    else:
        row["EAR_dev"] = row["EAR"]
        row["MAR_dev"] = row["MAR"]

    return row

features_df = features_df.apply(normalize, axis=1)

display(features_df.head())
features_df.to_csv("/kaggle/working/features_normalized.csv", index=False)

print("✅ Saved normalized features file")

,subject,label,frame,EAR,MAR,pitch,yaw,video_path,EAR_dev,MAR_dev
0,20,Low Vigilance,30,0.265168,0.093651,476.009454,748.818403,/kaggle/input/datasets/rishab260/uta-reallife-...,0.011799,0.084260
1,20,Low Vigilance,40,0.193316,0.003777,458.795161,741.967654,/kaggle/input/datasets/rishab260/uta-reallife-...,-0.060053,-0.005614
2,20,Low Vigilance,50,0.209924,0.021052,417.269697,681.898819,/kaggle/input/datasets/rishab260/uta-reallife-...,-0.043445,0.011661
3,20,Low Vigilance,60,0.200726,0.034570,373.262642,636.804523,/kaggle/input/datasets/rishab260/uta-reallife-...,-0.052643,0.025179
4,20,Low Vigilance,70,0.189407,0.006242,360.013889,629.407658,/kaggle/input/datasets/rishab260/uta-reallife-...,-0.063962,-0.003149


✅ Saved normalized features file


In [20]:
window_size = 15
windows = []

for subject in features_df["subject"].unique():
    sub_df = features_df[features_df["subject"] == subject].copy()
    sub_df = sub_df.sort_values(["video_path", "frame"])

    for video_path in sub_df["video_path"].unique():
        vid_df = sub_df[sub_df["video_path"] == video_path]

        for i in range(len(vid_df) - window_size):
            chunk = vid_df.iloc[i:i + window_size]

            windows.append({
                "subject": subject,
                "video_path": video_path,
                "label": chunk["label"].iloc[0],
                "EAR_mean": chunk["EAR"].mean(),
                "EAR_std": chunk["EAR"].std(),
                "EAR_min": chunk["EAR"].min(),
                "MAR_mean": chunk["MAR"].mean(),
                "MAR_max": chunk["MAR"].max(),
                "pitch_mean": chunk["pitch"].mean(),
                "yaw_mean": chunk["yaw"].mean(),
                "EAR_dev_mean": chunk["EAR_dev"].mean(),
                "MAR_dev_mean": chunk["MAR_dev"].mean()
            })

window_df = pd.DataFrame(windows)

print("Window rows:", len(window_df))
display(window_df.head())
window_df.to_csv("/kaggle/working/window_features.csv", index=False)

print("✅ Saved window features: /kaggle/working/window_features.csv")

Window rows: 233591


,subject,video_path,label,EAR_mean,EAR_std,EAR_min,MAR_mean,MAR_max,pitch_mean,yaw_mean,EAR_dev_mean,MAR_dev_mean
0,20,/kaggle/input/datasets/rishab260/uta-reallife-...,Alert,0.237036,0.037249,0.137103,0.036132,0.199047,370.103525,635.342636,-0.016333,0.026740
1,20,/kaggle/input/datasets/rishab260/uta-reallife-...,Alert,0.247254,0.045480,0.137103,0.035163,0.199047,332.952045,581.059837,-0.006115,0.025772
2,20,/kaggle/input/datasets/rishab260/uta-reallife-...,Alert,0.252945,0.050826,0.137103,0.029647,0.199047,327.099773,572.753631,-0.000425,0.020256
3,20,/kaggle/input/datasets/rishab260/uta-reallife-...,Alert,0.257431,0.055144,0.137103,0.030483,0.199047,314.021163,556.392134,0.004062,0.021092
4,20,/kaggle/input/datasets/rishab260/uta-reallife-...,Alert,0.256823,0.055100,0.137103,0.018152,0.042090,303.104617,530.404469,0.003454,0.008761


✅ Saved window features: /kaggle/working/window_features.csv


In [21]:
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

le = LabelEncoder()
window_df["y"] = le.fit_transform(window_df["label"])

drop_cols = ["label", "y", "subject", "video_path"]
X = window_df.drop(drop_cols, axis=1)
y = window_df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print(classification_report(
    y_test,
    pred,
    target_names=le.classes_
))

# Save model, label encoder, and feature columns
joblib.dump(model, "/kaggle/working/random_forest_model.pkl")
joblib.dump(le, "/kaggle/working/label_encoder.pkl")
joblib.dump(X.columns.tolist(), "/kaggle/working/feature_columns.pkl")

print("✅ Model saved: /kaggle/working/random_forest_model.pkl")
print("✅ Label encoder saved: /kaggle/working/label_encoder.pkl")
print("✅ Feature columns saved: /kaggle/working/feature_columns.pkl")

               precision    recall  f1-score   support

        Alert       1.00      1.00      1.00     15619
       Drowsy       1.00      1.00      1.00     15693
Low Vigilance       1.00      1.00      1.00     15407

     accuracy                           1.00     46719
    macro avg       1.00      1.00      1.00     46719
 weighted avg       1.00      1.00      1.00     46719

✅ Model saved: /kaggle/working/random_forest_model.pkl
✅ Label encoder saved: /kaggle/working/label_encoder.pkl
✅ Feature columns saved: /kaggle/working/feature_columns.pkl


In [22]:
report = classification_report(
    y_test,
    pred,
    target_names=le.classes_
)

print(report)

with open("/kaggle/working/classification_report.txt", "w") as f:
    f.write(report)

print("✅ Classification report saved")

               precision    recall  f1-score   support

        Alert       1.00      1.00      1.00     15619
       Drowsy       1.00      1.00      1.00     15693
Low Vigilance       1.00      1.00      1.00     15407

     accuracy                           1.00     46719
    macro avg       1.00      1.00      1.00     46719
 weighted avg       1.00      1.00      1.00     46719

✅ Classification report saved


In [23]:
pred = model.predict(X_test)

In [24]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, pred)

print("Accuracy:", accuracy)
print("Accuracy percentage:", accuracy * 100, "%")

Accuracy: 0.9962113915109484
Accuracy percentage: 99.62113915109484 %


In [25]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, pred)

print(f"✅ Model Accuracy: {accuracy:.4f}")
print(f"✅ Model Accuracy Percentage: {accuracy * 100:.2f}%")

✅ Model Accuracy: 0.9962
✅ Model Accuracy Percentage: 99.62%


In [26]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, pred)

accuracy_text = f"Model Accuracy: {accuracy:.4f}\nModel Accuracy Percentage: {accuracy * 100:.2f}%"

print(accuracy_text)

with open("/kaggle/working/accuracy.txt", "w") as f:
    f.write(accuracy_text)

print("✅ Accuracy saved at: /kaggle/working/accuracy.txt")

Model Accuracy: 0.9962
Model Accuracy Percentage: 99.62%
✅ Accuracy saved at: /kaggle/working/accuracy.txt


In [27]:
accuracy = model.score(X_test, y_test)

print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy Percentage: {accuracy * 100:.2f}%")

Accuracy: 0.9962
Accuracy Percentage: 99.62%


In [28]:
from sklearn.metrics import accuracy_score

pred = model.predict(X_test)

accuracy = accuracy_score(y_test, pred)

print("=" * 40)
print(f"✅ Final Test Accuracy: {accuracy:.4f}")
print(f"✅ Final Test Accuracy Percentage: {accuracy * 100:.2f}%")
print("=" * 40)

✅ Final Test Accuracy: 0.9962
✅ Final Test Accuracy Percentage: 99.62%


In [30]:
import pandas as pd

features_df = pd.read_csv("/kaggle/working/features_progress.csv")

numeric_cols = ["EAR", "MAR", "pitch", "yaw"]

baseline = (
    features_df[features_df["label"] == "Alert"]
    .groupby("subject")[numeric_cols]
    .mean()
)

baseline.to_csv("/kaggle/working/baseline_by_subject.csv")

print("✅ Baseline saved at: /kaggle/working/baseline_by_subject.csv")
display(baseline.head())

✅ Baseline saved at: /kaggle/working/baseline_by_subject.csv


,EAR,MAR,pitch,yaw
subject,,,,
1,0.303734,0.010927,230.098915,369.902234
2,0.301671,0.014962,205.900822,324.892506
3,0.283038,0.010225,203.785617,368.570744
4,0.296292,0.013034,97.361923,160.952923
5,0.327134,0.012007,194.941465,308.112217


In [31]:
import joblib
import pandas as pd
import numpy as np
from collections import Counter

model = joblib.load("/kaggle/working/random_forest_model.pkl")
le = joblib.load("/kaggle/working/label_encoder.pkl")
feature_columns = joblib.load("/kaggle/working/feature_columns.pkl")

baseline = pd.read_csv("/kaggle/working/baseline_by_subject.csv", index_col=0)

print("✅ Model loaded")
print("Classes:", le.classes_)
print("Feature columns:", feature_columns)

✅ Model loaded
Classes: ['Alert' 'Drowsy' 'Low Vigilance']
Feature columns: ['EAR_mean', 'EAR_std', 'EAR_min', 'MAR_mean', 'MAR_max', 'pitch_mean', 'yaw_mean', 'EAR_dev_mean', 'MAR_dev_mean']


In [32]:
def normalize_test_features(features_df, subject):
    """
    Adds EAR_dev and MAR_dev to test video features.
    If subject exists in training baseline, use that baseline.
    Otherwise use raw EAR and MAR as fallback.
    """

    features_df = features_df.copy()

    if subject in baseline.index:
        features_df["EAR_dev"] = features_df["EAR"] - baseline.loc[subject, "EAR"]
        features_df["MAR_dev"] = features_df["MAR"] - baseline.loc[subject, "MAR"]
        print(f"✅ Used baseline for subject: {subject}")
    else:
        features_df["EAR_dev"] = features_df["EAR"]
        features_df["MAR_dev"] = features_df["MAR"]
        print(f"⚠️ No baseline found for subject: {subject}. Using raw EAR/MAR.")

    return features_df

In [33]:
def create_test_windows(features_df, window_size=15):
    windows = []

    features_df = features_df.sort_values(["video_path", "frame"])

    for video_path in features_df["video_path"].unique():
        vid_df = features_df[features_df["video_path"] == video_path]

        if len(vid_df) < window_size:
            print(f"❌ Not enough frames. Need at least {window_size}, got {len(vid_df)}")
            return pd.DataFrame()

        for i in range(len(vid_df) - window_size + 1):
            chunk = vid_df.iloc[i:i + window_size]

            windows.append({
                "EAR_mean": chunk["EAR"].mean(),
                "EAR_std": chunk["EAR"].std(),
                "EAR_min": chunk["EAR"].min(),
                "MAR_mean": chunk["MAR"].mean(),
                "MAR_max": chunk["MAR"].max(),
                "pitch_mean": chunk["pitch"].mean(),
                "yaw_mean": chunk["yaw"].mean(),
                "EAR_dev_mean": chunk["EAR_dev"].mean(),
                "MAR_dev_mean": chunk["MAR_dev"].mean()
            })

    window_df = pd.DataFrame(windows)

    return window_df

In [34]:
def predict_video(video_path, subject="test_subject", sample_rate=10, window_size=15):
    """
    Predicts Alert / Low Vigilance / Drowsy for a new video.
    """

    print("Processing video:", video_path)

    rows = process_video(
        path=video_path,
        label="Unknown",
        subject=subject,
        sample_rate=sample_rate
    )

    frame_features = pd.DataFrame(rows)

    if len(frame_features) == 0:
        print("❌ No face/features extracted from video.")
        return None

    print("Extracted frame features:", len(frame_features))

    frame_features = normalize_test_features(frame_features, subject)

    window_features = create_test_windows(
        frame_features,
        window_size=window_size
    )

    if len(window_features) == 0:
        print("❌ No windows created.")
        return None

    print("Created windows:", len(window_features))

    X_new = window_features[feature_columns]

    pred_encoded = model.predict(X_new)
    pred_labels = le.inverse_transform(pred_encoded)

    window_features["prediction"] = pred_labels

    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_new)
        avg_probs = probs.mean(axis=0)

        prob_df = pd.DataFrame({
            "class": le.classes_,
            "average_probability": avg_probs
        }).sort_values("average_probability", ascending=False)

        print("\nAverage class probabilities:")
        display(prob_df)

    counts = Counter(pred_labels)

    print("\nWindow prediction counts:")
    for cls, count in counts.items():
        print(f"{cls}: {count}")

    final_prediction = counts.most_common(1)[0][0]

    print("\n" + "=" * 40)
    print(f"✅ Final Video Prediction: {final_prediction}")
    print("=" * 40)

    return final_prediction, window_features

In [35]:
test_video_path = "/kaggle/input/datasets/rishab260/uta-reallife-drowsiness-dataset/10_1.mp4"

final_prediction, prediction_windows = predict_video(
    video_path=test_video_path,
    subject="test_subject",
    sample_rate=10,
    window_size=15
)

Processing video: /kaggle/input/datasets/rishab260/uta-reallife-drowsiness-dataset/10_1.mp4
Could not open video: /kaggle/input/datasets/rishab260/uta-reallife-drowsiness-dataset/10_1.mp4
❌ No face/features extracted from video.


TypeError: cannot unpack non-iterable NoneType object

In [37]:
df = pd.read_csv("/kaggle/working/metadata_clean.csv")

sample_video = df.iloc[0]["path"]
sample_subject = df.iloc[0]["subject"]

print(sample_video)
print(sample_subject)

final_prediction, prediction_windows = predict_video(
    video_path=sample_video,
    subject=sample_subject,
    sample_rate=10,
    window_size=15
)

/kaggle/input/datasets/rishab260/uta-reallife-drowsiness-dataset/Fold4_part2/Fold4_part2/47/5.mp4
47
Processing video: /kaggle/input/datasets/rishab260/uta-reallife-drowsiness-dataset/Fold4_part2/Fold4_part2/47/5.mp4
Done: 5.mp4 | processed=1802, saved=1802, failed=0
Extracted frame features: 1802
✅ Used baseline for subject: 47
Created windows: 1788

Average class probabilities:


,class,average_probability
2,Low Vigilance,0.976773
0,Alert,0.011714
1,Drowsy,0.011513



Window prediction counts:
Low Vigilance: 1787
Alert: 1

✅ Final Video Prediction: Low Vigilance


In [38]:
display(prediction_windows.head())
prediction_windows["prediction"].value_counts()
prediction_windows.to_csv("/kaggle/working/test_video_predictions.csv", index=False)

print("✅ Saved predictions at: /kaggle/working/test_video_predictions.csv")

,EAR_mean,EAR_std,EAR_min,MAR_mean,MAR_max,pitch_mean,yaw_mean,EAR_dev_mean,MAR_dev_mean,prediction
0,0.309336,0.023248,0.28351,0.007876,0.012619,231.100372,387.042458,-0.027779,0.002260,Low Vigilance
1,0.312010,0.023875,0.28351,0.008456,0.012619,229.385464,383.802663,-0.025105,0.002839,Low Vigilance
2,0.313832,0.022875,0.28351,0.008511,0.012619,228.005915,381.044657,-0.023283,0.002894,Low Vigilance
3,0.315134,0.021684,0.28351,0.009048,0.012619,226.347432,378.087236,-0.021981,0.003432,Low Vigilance
4,0.316458,0.021064,0.28351,0.009089,0.012619,224.860678,376.021265,-0.020657,0.003472,Low Vigilance


✅ Saved predictions at: /kaggle/working/test_video_predictions.csv


In [ ]:
df = pd.read_csv("/kaggle/working/metadata_clean.csv")

test_index = 0

test_video_path = df.iloc[test_index]["path"]
test_subject = df.iloc[test_index]["subject"]
true_label = df.iloc[test_index]["label"]

print("Video:", test_video_path)
print("Subject:", test_subject)
print("True label:", true_label)

final_prediction, prediction_windows = predict_video(
    video_path=test_video_path,
    subject=test_subject,
    sample_rate=10,
    window_size=15
)

print("True label:", true_label)
print("Predicted label:", final_prediction)

In [ ]:
import cv2
import numpy as np
import pandas as pd
import joblib
import mediapipe as mp
from collections import deque, Counter

from mediapipe.tasks import python
from mediapipe.tasks.python import vision


# =========================
# 1. LOAD SAVED MODEL FILES
# =========================

model = joblib.load("random_forest_model.pkl")
le = joblib.load("label_encoder.pkl")
feature_columns = joblib.load("feature_columns.pkl")

print("✅ Model loaded")
print("Classes:", le.classes_)
print("Feature columns:", feature_columns)


# =========================
# 2. LOAD MEDIAPIPE MODEL
# =========================

base_options = python.BaseOptions(
    model_asset_path="face_landmarker.task"
)

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False
)

detector = vision.FaceLandmarker.create_from_options(options)

print("✅ FaceLandmarker ready")


# =========================
# 3. LANDMARK DEFINITIONS
# =========================

# Eye landmarks for EAR
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

# Mouth landmarks for MAR
MOUTH_TOP = 13
MOUTH_BOTTOM = 14
MOUTH_LEFT = 78
MOUTH_RIGHT = 308

# Approximate head pose landmarks
NOSE = 1
CHIN = 152
LEFT_FACE = 234
RIGHT_FACE = 454


# =========================
# 4. FEATURE FUNCTIONS
# =========================

def compute_EAR(landmarks, eye):
    """
    EAR = Eye Aspect Ratio.
    Smaller EAR means eyes are more closed.
    """
    p1, p2, p3, p4, p5, p6 = [np.array(landmarks[i][:2]) for i in eye]

    vertical1 = np.linalg.norm(p2 - p6)
    vertical2 = np.linalg.norm(p3 - p5)
    horizontal = np.linalg.norm(p1 - p4)

    return (vertical1 + vertical2) / (2.0 * horizontal + 1e-6)


def compute_MAR(landmarks):
    """
    MAR = Mouth Aspect Ratio.
    Larger MAR can indicate open mouth/yawning.
    """
    top = np.array(landmarks[MOUTH_TOP][:2])
    bottom = np.array(landmarks[MOUTH_BOTTOM][:2])
    left = np.array(landmarks[MOUTH_LEFT][:2])
    right = np.array(landmarks[MOUTH_RIGHT][:2])

    vertical = np.linalg.norm(top - bottom)
    horizontal = np.linalg.norm(left - right)

    return vertical / (horizontal + 1e-6)


def compute_head_pose_approx(landmarks):
    """
    Approximate head pose features.
    Same as your training code.
    """
    nose = np.array(landmarks[NOSE][:2])
    chin = np.array(landmarks[CHIN][:2])
    left = np.array(landmarks[LEFT_FACE][:2])
    right = np.array(landmarks[RIGHT_FACE][:2])

    pitch = np.linalg.norm(nose - chin)
    yaw = np.linalg.norm(left - right)

    return pitch, yaw


def get_face_landmarks_from_frame(frame_bgr):
    """
    Input: OpenCV BGR frame
    Output: list of pixel landmarks [(x, y, z), ...] or None
    """

    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = detector.detect(mp_image)

    if not result.face_landmarks:
        return None

    h, w, _ = frame_bgr.shape

    landmarks_px = []

    for lm in result.face_landmarks[0]:
        x = int(lm.x * w)
        y = int(lm.y * h)
        z = lm.z
        landmarks_px.append((x, y, z))

    return landmarks_px


def extract_features_from_frame(frame):
    """
    Extract EAR, MAR, pitch, yaw from one webcam frame.
    """

    landmarks = get_face_landmarks_from_frame(frame)

    if landmarks is None:
        return None

    try:
        left_ear = compute_EAR(landmarks, LEFT_EYE)
        right_ear = compute_EAR(landmarks, RIGHT_EYE)
        EAR = (left_ear + right_ear) / 2.0

        MAR = compute_MAR(landmarks)
        pitch, yaw = compute_head_pose_approx(landmarks)

        return {
            "EAR": EAR,
            "MAR": MAR,
            "pitch": pitch,
            "yaw": yaw
        }

    except Exception as e:
        print("Feature extraction error:", e)
        return None


# =========================
# 5. WINDOW FEATURE FUNCTION
# =========================

def create_window_features(buffer, baseline_EAR, baseline_MAR):
    """
    Convert last 15 frame-level features into one row for model prediction.
    This must match your training features.
    """

    df = pd.DataFrame(list(buffer))

    df["EAR_dev"] = df["EAR"] - baseline_EAR
    df["MAR_dev"] = df["MAR"] - baseline_MAR

    row = {
        "EAR_mean": df["EAR"].mean(),
        "EAR_std": df["EAR"].std(),
        "EAR_min": df["EAR"].min(),
        "MAR_mean": df["MAR"].mean(),
        "MAR_max": df["MAR"].max(),
        "pitch_mean": df["pitch"].mean(),
        "yaw_mean": df["yaw"].mean(),
        "EAR_dev_mean": df["EAR_dev"].mean(),
        "MAR_dev_mean": df["MAR_dev"].mean()
    }

    X = pd.DataFrame([row])

    # Ensure same column order as training
    X = X[feature_columns]

    return X


# =========================
# 6. MAIN REAL-TIME FUNCTION
# =========================

def run_realtime_prediction(
    webcam_index=0,
    window_size=15,
    calibration_frames=60,
    sample_every_n_frames=3,
    smoothing_window=10
):
    """
    Real-time prediction from webcam.

    calibration_frames:
        Number of valid face frames used to calculate EAR/MAR baseline.

    sample_every_n_frames:
        Process every Nth webcam frame.
        If webcam is slow, use 2 or 3.
        If CPU is weak, use 5.

    smoothing_window:
        Majority voting over last N predictions.
    """

    cap = cv2.VideoCapture(webcam_index)

    if not cap.isOpened():
        print("❌ Could not open webcam")
        return

    print("✅ Webcam opened")
    print("Calibration started...")
    print("Please look normal/alert at the camera.")

    frame_count = 0
    calibration_data = []

    # =========================
    # CALIBRATION PHASE
    # =========================

    while len(calibration_data) < calibration_frames:
        ret, frame = cap.read()

        if not ret:
            print("❌ Failed to read webcam frame")
            break

        frame = cv2.flip(frame, 1)
        frame_count += 1

        display_frame = frame.copy()

        cv2.putText(
            display_frame,
            f"Calibration: {len(calibration_data)}/{calibration_frames}",
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            2
        )

        cv2.putText(
            display_frame,
            "Stay ALERT and look at camera",
            (30, 90),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 255),
            2
        )

        if frame_count % sample_every_n_frames == 0:
            features = extract_features_from_frame(frame)

            if features is not None:
                calibration_data.append(features)

        cv2.imshow("Real-Time Drowsiness Detection", display_frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            cap.release()
            cv2.destroyAllWindows()
            return

    if len(calibration_data) == 0:
        print("❌ Calibration failed. No face detected.")
        cap.release()
        cv2.destroyAllWindows()
        return

    calibration_df = pd.DataFrame(calibration_data)

    baseline_EAR = calibration_df["EAR"].mean()
    baseline_MAR = calibration_df["MAR"].mean()

    print("✅ Calibration completed")
    print("Baseline EAR:", baseline_EAR)
    print("Baseline MAR:", baseline_MAR)

    # =========================
    # PREDICTION PHASE
    # =========================

    feature_buffer = deque(maxlen=window_size)
    prediction_buffer = deque(maxlen=smoothing_window)

    current_prediction = "Collecting frames..."
    current_confidence = 0.0

    print("✅ Real-time prediction started")
    print("Press 'q' to quit")

    while True:
        ret, frame = cap.read()

        if not ret:
            print("❌ Failed to read frame")
            break

        frame = cv2.flip(frame, 1)
        frame_count += 1

        display_frame = frame.copy()

        if frame_count % sample_every_n_frames == 0:
            features = extract_features_from_frame(frame)

            if features is not None:
                feature_buffer.append(features)

                if len(feature_buffer) == window_size:
                    X_live = create_window_features(
                        feature_buffer,
                        baseline_EAR,
                        baseline_MAR
                    )

                    pred_encoded = model.predict(X_live)[0]
                    pred_label = le.inverse_transform([pred_encoded])[0]

                    prediction_buffer.append(pred_label)

                    # Majority vote smoothing
                    current_prediction = Counter(prediction_buffer).most_common(1)[0][0]

                    if hasattr(model, "predict_proba"):
                        probs = model.predict_proba(X_live)[0]
                        current_confidence = np.max(probs) * 100

        # Choose color based on prediction
        if current_prediction == "Alert":
            color = (0, 255, 0)
        elif current_prediction == "Low Vigilance":
            color = (0, 255, 255)
        elif current_prediction == "Drowsy":
            color = (0, 0, 255)
        else:
            color = (255, 255, 255)

        # Display prediction
        cv2.putText(
            display_frame,
            f"Prediction: {current_prediction}",
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            color,
            2
        )

        cv2.putText(
            display_frame,
            f"Confidence: {current_confidence:.2f}%",
            (30, 90),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            color,
            2
        )

        cv2.putText(
            display_frame,
            f"Buffer: {len(feature_buffer)}/{window_size}",
            (30, 130),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

        cv2.putText(
            display_frame,
            "Press q to quit",
            (30, display_frame.shape[0] - 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

        cv2.imshow("Real-Time Drowsiness Detection", display_frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("✅ Webcam closed")


# =========================
# 7. RUN
# =========================

if __name__ == "__main__":
    run_realtime_prediction(
        webcam_index=0,
        window_size=15,
        calibration_frames=60,
        sample_every_n_frames=3,
        smoothing_window=10
    )